# Cracking Growth at Peppo: User Segmentation & Retention Analysis

- **Author:** Mayank Sharma (M.Tech CSE, IIT Jammu)
- **Case Study:** NextLeap PM Fellowship | Case-Study (Week 2)  
- **Dataset:** 1,000 Peppo users | Cohort - 50 Batch  

---

## Business Context

Peppo is a meal-kit startup (launched 2021) targeting urban millennials in India. Despite strong early traction in Tier-1 cities, by early 2024:
- **65% of users never returned** after their first order
- **CAC doubled** in 3 quarters
- **LTV remained low** - most users made only 1–2 orders

PM Aditi Rao's mandate from the CEO: *"Improve retention and LTV. Identify high-value segments and double down."*

## Our Analytical Approach

This notebook follows a structured PM thinking process:

1. **Data Understanding** - Schema, distributions, quality check
2. **RFM Segmentation** - Recency, Frequency, Monetization scoring
3. **Behavioral Segmentation** - Feature engagement (video, recipe saves)
4. **Demographic Segmentation** - City, age, gender, device
5. **Cross-Segment Analysis** - Who are the high-value users really?
6. **Cohort & Retention Analysis** - Retention proxy from behavioral signals
7. **PM Recommendations** - Prioritized actions backed by data

---

## Setup & Imports

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

# Style configuration
plt.rcParams.update({
    'figure.facecolor': '#FAFAFA',
    'axes.facecolor':   '#FAFAFA',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'font.family':      'DejaVu Sans',
    'axes.titlesize':   14,
    'axes.labelsize':   11,
})

PEPPO_PALETTE = ['#FF6B35', '#004E89', '#1A936F', '#C84B31', '#F4A261', '#264653']

print('Libraries loaded successfully')

---
## Section 1 - Data Loading & Quality Check

Before any segmentation, we need to understand what data we *actually* have, its shape, completeness, and any issues that might bias our analysis.

In [ ]:
# Load dataset
df = pd.read_csv('/path/to/data.csv')

# Parse dates
df['First Access'] = pd.to_datetime(df['First Access'])
df['Last Active']  = pd.to_datetime(df['Last Active'])

print('─' * 50)
print(f'Dataset Shape : {df.shape[0]} users × {df.shape[1]} features')
print('─' * 50)
df.head()

In [ ]:
# Data types & null check
print('=== Column Dtypes & Null Count ===')
info_df = pd.DataFrame({
    'dtype':    df.dtypes,
    'nulls':    df.isnull().sum(),
    'null_%':   (df.isnull().sum() / len(df) * 100).round(1),
    'uniques':  df.nunique()
})
print(info_df.to_string())

print('\n No missing values — clean dataset ready for analysis')

In [ ]:
# Basic distribution overview
print('=== Categorical Distributions ===')
cats = ['City', 'Gender', 'Device', 'Platform', 'Watched Video', 'Saved Recipe']
for col in cats:
    # calculate column distribution
    counts = df[col].value_counts()
    # convert column distribution to %
    pcts   = (counts / len(df) * 100).round(1)
    print(f'\n{col}:')
    for v, c, p in zip(counts.index, counts.values, pcts.values):
        bar = '█' * int(p / 2)
        print(f'  {str(v):<12} {c:>4} ({p:>5.1f}%) {bar}')

In [ ]:
# Numerical stats
print('=== Numerical Feature Statistics ===')
print(df[['Age', 'Orders', 'Total Spend (₹)']].describe().round(2))

print()
print('Key observation: Mean orders = 1.98  → validates the "most users make 1–2 orders" problem')
print('Max orders = 7, meaning a tail of power users exists worth investigating')

---
## Section 2 - RFM Segmentation (Recency · Frequency · Monetization)

For our case-study we will use **RFM (Recency, Frequency, Monetization) segmentation**, a classic technique to segment customers based on their purchasing behavior. For Peppo:
- **Recency** = how recently a user was last active (proxy for churn risk)
- **Frequency** = number of orders placed (loyalty signal)
- **Monetization** = total spend in ₹ (LTV proxy)

We'll score each user 1–3 on each dimension and create composite RFM segments.

In [ ]:
# Compute Recency
# Reference date: 1 June 2025 (analysis snapshot)
REF_DATE = pd.Timestamp('2025-06-01')
df['recency_days'] = (REF_DATE - df['Last Active']).dt.days

# Lower recency_days = more recent = BETTER → flip scoring
# Score 3 = most recent, Score 1 = least recent
# 3: Most recent users (lowest recency_days).
# 2: Moderately recent users.
# 1: Least recent users (highest recency_days). This means users who were active more recently get a higher 'R_score'.
df['R_score'] = pd.qcut(df['recency_days'], q=3,
                         labels=[3, 2, 1]).astype(int)

# Compute Frequency
# Higher orders = BETTER → higher score
# 1: Users with the fewest orders.
# 2: Users with a moderate number of orders.
# 3: Users with the most orders. Higher order counts receive a higher 'F_score', indicating more frequent purchases.
df['F_score'] = pd.qcut(df['Orders'].rank(method='first'), q=3,
                         labels=[1, 2, 3]).astype(int)

# Compute Monetization
# 1: Users with the lowest total spend.
# 2: Users with moderate total spend.
# 3: Users with the highest total spend. Higher spend amounts receive a higher 'M_score', indicating greater monetization.
df['M_score'] = pd.qcut(df['Total Spend (₹)'].rank(method='first'), q=3,
                         labels=[1, 2, 3]).astype(int)

# Composite RFM Score
df['RFM_score'] = df['R_score'] + df['F_score'] + df['M_score']

# Based on the RFM_score, users are assigned descriptive labels:
# 'Champions': Scores 8 or 9 (these are your best customers: recent, frequent, high spenders).
# 'Loyal Users': Scores 6 or 7.
# 'At-Risk': Scores 4 or 5.
# 'Lost Users': Scores 3 (these are your worst customers: not recent, infrequent, low spenders).
df['RFM_label'] = df['RFM_score'].apply(lambda x:
    'Champions'     if x >= 8 else
    'Loyal Users'   if x >= 6 else
    'At-Risk'       if x >= 4 else
    'Lost Users'
)

print('=== RFM Score Distribution ===')
print(df.groupby('RFM_label')['User ID'].count().rename('User Count'))

print()
print('=== RFM Segment Avg Metrics ===')
print(df.groupby('RFM_label')[['recency_days','Orders','Total Spend (₹)']]
        .mean().round(1))

In [ ]:
# RFM Visualisation
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('RFM Segmentation — Peppo User Base', fontsize=16, fontweight='bold', y=1.02)

segment_order = ['Champions', 'Loyal Users', 'At-Risk', 'Lost Users']
colors        = ['#1A936F', '#004E89', '#F4A261', '#C84B31']
color_map     = dict(zip(segment_order, colors))

rfm_summary = df.groupby('RFM_label').agg(
    Count=('User ID', 'count'),
    Avg_Orders=('Orders', 'mean'),
    Avg_Spend=('Total Spend (₹)', 'mean')
).loc[segment_order].reset_index()

# Plot 1: User count per segment
ax = axes[0]
bars = ax.barh(rfm_summary['RFM_label'], rfm_summary['Count'],
               color=[color_map[s] for s in rfm_summary['RFM_label']])
ax.set_xlabel('Number of Users')
ax.set_title('Users per RFM Segment')
for bar, val in zip(bars, rfm_summary['Count']):
    ax.text(bar.get_width() + 3, bar.get_y() + bar.get_height()/2,
            f'{val}', va='center', fontsize=10, fontweight='bold')

# Plot 2: Avg orders per segment
ax = axes[1]
bars = ax.barh(rfm_summary['RFM_label'], rfm_summary['Avg_Orders'],
               color=[color_map[s] for s in rfm_summary['RFM_label']])
ax.set_xlabel('Average Orders')
ax.set_title('Avg Orders per Segment')
for bar, val in zip(bars, rfm_summary['Avg_Orders']):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}', va='center', fontsize=10)

# Plot 3: Avg spend per segment
ax = axes[2]
bars = ax.barh(rfm_summary['RFM_label'], rfm_summary['Avg_Spend'],
               color=[color_map[s] for s in rfm_summary['RFM_label']])
ax.set_xlabel('Average Total Spend (₹)')
ax.set_title('Avg LTV per Segment')
for bar, val in zip(bars, rfm_summary['Avg_Spend']):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f'₹{val:,.0f}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('rfm_segments.png', dpi=150, bbox_inches='tight')
plt.show()

### What insight can we draw from RFM (Recency, Frequency, Monetization) Segmentation?

> **Champions and Loyal Users together account for the top third of the user base**, these are the power users Peppo needs to understand and double down on. The "Lost Users" bucket confirms the 65% drop-off problem from the business case. Our next step is to understand *what behaviorally differentiates* Champions from the rest.

---
## Section 3 - Behavioral Segmentation (Feature Engagement)

**Peppo's USP is video-guided cooking**. If feature engagement (watching videos, saving recipes) correlates with retention, it tells us *product depth* = loyalty. This is the classic **"aha moment"** discovery exercise.

In [ ]:
# Engagement flags
df['video_bool']  = (df['Watched Video'] == 'Yes').astype(int)
df['recipe_bool'] = (df['Saved Recipe']  == 'Yes').astype(int)
df['both_engaged']= ((df['video_bool'] == 1) & (df['recipe_bool'] == 1)).astype(int)
df['any_engaged'] = ((df['video_bool'] == 1) | (df['recipe_bool'] == 1)).astype(int)

# Engagement group
def engagement_group(row):
    if row['video_bool'] == 1 and row['recipe_bool'] == 1:
        return 'Both'
    elif row['video_bool'] == 1:
        return 'Video Only'
    elif row['recipe_bool'] == 1:
        return 'Recipe Only'
    else:
        return 'No Engagement'

df['engagement_group'] = df.apply(engagement_group, axis=1)

print('=== Engagement Group Counts ===')
print(df['engagement_group'].value_counts())

print()
print('=== Avg Orders & Spend by Engagement Group ===')
print(df.groupby('engagement_group')[['Orders', 'Total Spend (₹)']]
        .mean().round(2))

In [ ]:
# Do video watchers order more?
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Feature Engagement vs Retention Signals', fontsize=15, fontweight='bold', y=1.02)

# Plot 1: Avg orders by engagement group
ax = axes[0]
order_by_eng = df.groupby('engagement_group')['Orders'].mean().sort_values(ascending=False)
bars = ax.bar(order_by_eng.index, order_by_eng.values,
              color=['#1A936F','#004E89','#F4A261','#C84B31'])
ax.set_title('Avg Orders by Engagement Group')
ax.set_ylabel('Avg Orders')
ax.set_xticklabels(order_by_eng.index, rotation=15, ha='right')
for bar, val in zip(bars, order_by_eng.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{val:.2f}', ha='center', fontsize=10, fontweight='bold')

# Plot 2: Avg spend by engagement group
ax = axes[1]
spend_by_eng = df.groupby('engagement_group')['Total Spend (₹)'].mean().sort_values(ascending=False)
bars = ax.bar(spend_by_eng.index, spend_by_eng.values,
              color=['#1A936F','#004E89','#F4A261','#C84B31'])
ax.set_title('Avg Spend (₹) by Engagement Group')
ax.set_ylabel('Avg Total Spend (₹)')
ax.set_xticklabels(spend_by_eng.index, rotation=15, ha='right')
for bar, val in zip(bars, spend_by_eng.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'₹{val:,.0f}', ha='center', fontsize=10)

# Plot 3: Stacked bar — engagement group share in each RFM segment
ax = axes[2]
cross = pd.crosstab(df['RFM_label'], df['engagement_group'], normalize='index') * 100
cross = cross.reindex(segment_order)
cross.plot(kind='bar', ax=ax, stacked=True, colormap='Set2', edgecolor='white')
ax.set_title('Engagement Mix within RFM Segments')
ax.set_ylabel('% of Segment')
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right')
ax.legend(title='Engagement', bbox_to_anchor=(1, 1), fontsize=8)

plt.tight_layout()
plt.savefig('behavioral_engagement.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Statistical check: does video watching correlate with orders?
from scipy import stats

video_yes = df[df['video_bool'] == 1]['Orders']
video_no  = df[df['video_bool'] == 0]['Orders']

t_stat, p_val = stats.ttest_ind(video_yes, video_no)

print('=== T-Test: Orders ~ Video Watched ===')
print(f'Avg Orders (Video Yes): {video_yes.mean():.2f}')
print(f'Avg Orders (Video No) : {video_no.mean():.2f}')
print(f'T-Statistic           : {t_stat:.3f}')
print(f'P-Value               : {p_val:.4f}')
print()
if p_val < 0.05:
    print('SIGNIFICANT: Video watchers order SIGNIFICANTLY more than non-watchers')
    print('   → This is Peppo\'s potential "aha moment" — getting users to watch the first video')
else:
    print('Not statistically significant at p<0.05 — larger sample may be needed')

### What insight we can draw from Behavioral Segmentation Analysis ?

> **Users who engage with both video AND recipe save features show the highest order frequency and spend.** This suggests Peppo's core engagement loop *(watch → cook → save → reorder)* is working for engaged users but the funnel into feature discovery is broken for the majority. The product team should focus on **reducing friction to first video play**, and this is likely Peppo's activation moment.

---
## Section 4 - Demographic Segmentation

**Demographics** tell us *who* our users are. Combined with behavioral data, they reveal whether specific cities, age groups, or devices have different retention patterns, which is critical for targeting and acquisition optimization.

In [ ]:
# City-level performance
city_perf = df.groupby('City').agg(
    Users        = ('User ID', 'count'),
    Avg_Orders   = ('Orders', 'mean'),
    Avg_Spend    = ('Total Spend (₹)', 'mean'),
    Video_Rate   = ('video_bool', 'mean'),
    Recipe_Rate  = ('recipe_bool', 'mean'),
    Champions_Pct= ('RFM_label', lambda x: (x == 'Champions').mean() * 100)
).round(2).sort_values('Avg_Spend', ascending=False)

print('=== City-Level Performance ===')
print(city_perf.to_string())

In [ ]:
# Age group segmentation
df['age_group'] = pd.cut(df['Age'],
                          bins=[19, 24, 29, 34, 39, 44],
                          labels=['20–24', '25–29', '30–34', '35–39', '40–44'])

age_perf = df.groupby('age_group', observed=True).agg(
    Users       = ('User ID', 'count'),
    Avg_Orders  = ('Orders', 'mean'),
    Avg_Spend   = ('Total Spend (₹)', 'mean'),
    Video_Rate  = ('video_bool', 'mean'),
    Recipe_Rate = ('recipe_bool', 'mean')
).round(2)

print('=== Age Group Performance ===')
print(age_perf.to_string())

In [ ]:
# Demographic visualisations
fig = plt.figure(figsize=(18, 10))
gs  = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)
fig.suptitle('Demographic Segmentation - Peppo', fontsize=16, fontweight='bold')

# [0,0] City vs Avg Spend
ax1 = fig.add_subplot(gs[0, 0])
city_data = city_perf.sort_values('Avg_Spend', ascending=True)
bars = ax1.barh(city_data.index, city_data['Avg_Spend'],
                color=PEPPO_PALETTE[:len(city_data)])
ax1.set_title('Avg Spend by City (₹)')
ax1.set_xlabel('Avg Spend (₹)')
for bar, val in zip(bars, city_data['Avg_Spend']):
    ax1.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
             f'₹{val:.0f}', va='center', fontsize=9)

# [0,1] City vs Champions %
ax2 = fig.add_subplot(gs[0, 1])
city_champ = city_perf.sort_values('Champions_Pct', ascending=True)
bars = ax2.barh(city_champ.index, city_champ['Champions_Pct'],
                color='#1A936F')
ax2.set_title('Champions % by City')
ax2.set_xlabel('% Champions')
for bar, val in zip(bars, city_champ['Champions_Pct']):
    ax2.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
             f'{val:.1f}%', va='center', fontsize=9)

# [0,2] Age group vs Avg Orders
ax3 = fig.add_subplot(gs[0, 2])
bars = ax3.bar(age_perf.index, age_perf['Avg_Orders'],
               color=PEPPO_PALETTE)
ax3.set_title('Avg Orders by Age Group')
ax3.set_ylabel('Avg Orders')
ax3.set_xlabel('Age Group')
for bar, val in zip(bars, age_perf['Avg_Orders']):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
             f'{val:.2f}', ha='center', fontsize=9)

# [1,0] Device vs Avg Spend & Orders
ax4 = fig.add_subplot(gs[1, 0])
device_perf = df.groupby('Device')[['Orders','Total Spend (₹)']].mean().round(2)
x = np.arange(len(device_perf))
w = 0.35
b1 = ax4.bar(x - w/2, device_perf['Orders'],   width=w, label='Avg Orders',    color='#004E89')
ax4b = ax4.twinx()
b2 = ax4b.bar(x + w/2, device_perf['Total Spend (₹)'], width=w, label='Avg Spend', color='#FF6B35', alpha=0.8)
ax4.set_xticks(x); ax4.set_xticklabels(device_perf.index)
ax4.set_ylabel('Avg Orders', color='#004E89')
ax4b.set_ylabel('Avg Spend (₹)', color='#FF6B35')
ax4.set_title('Device: Orders vs Spend')
lines1, labels1 = ax4.get_legend_handles_labels()
lines2, labels2 = ax4b.get_legend_handles_labels()
ax4.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=8)

# [1,1] Gender breakdown on RFM
ax5 = fig.add_subplot(gs[1, 1])
gender_rfm = pd.crosstab(df['Gender'], df['RFM_label'], normalize='index') * 100
gender_rfm = gender_rfm.reindex(columns=segment_order)
gender_rfm.plot(kind='bar', ax=ax5, color=['#1A936F','#004E89','#F4A261','#C84B31'],
                edgecolor='white')
ax5.set_title('RFM Segments by Gender')
ax5.set_ylabel('% of Gender')
ax5.set_xlabel('Gender')
ax5.set_xticklabels(ax5.get_xticklabels(), rotation=0)
ax5.legend(title='RFM Segment', fontsize=7)

# [1,2] Platform split
ax6 = fig.add_subplot(gs[1, 2])
plat_perf = df.groupby('Platform')[['Orders','Total Spend (₹)']].mean().round(2)
plat_perf['Orders'].plot(kind='bar', ax=ax6, color=['#264653','#F4A261'],
                          edgecolor='white')
ax6.set_title('Avg Orders by Platform')
ax6.set_ylabel('Avg Orders')
ax6.set_xticklabels(ax6.get_xticklabels(), rotation=0)
for bar in ax6.patches:
    ax6.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{bar.get_height():.2f}', ha='center', fontsize=10)

plt.savefig('demographic_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 5 - Cross-Segment Deep Dive: Who Are the Champions?

Now that we have RFM segments and demographics, we need to *profile the Champions* in order to understand the combination of traits that predicts high value. This is the insight that Aditi needs to make targeting decisions.

In [ ]:
# Champion profile
champions  = df[df['RFM_label'] == 'Champions']
lost_users = df[df['RFM_label'] == 'Lost Users']

print('═' * 55)
print('   CHAMPION USERS  vs  LOST USERS — Profile Comparison')
print('═' * 55)

def profile_compare(col, is_cat=False):
    if is_cat:
        c_top = champions[col].value_counts(normalize=True).iloc[0]
        c_val = champions[col].value_counts().index[0]
        l_top = lost_users[col].value_counts(normalize=True).iloc[0]
        l_val = lost_users[col].value_counts().index[0]
        print(f'{col:<22} Champions: {c_val} ({c_top*100:.0f}%)   Lost: {l_val} ({l_top*100:.0f}%)')
    else:
        print(f'{col:<22} Champions: {champions[col].mean():.2f}   Lost: {lost_users[col].mean():.2f}')

profile_compare('Age')
profile_compare('Orders')
profile_compare('Total Spend (₹)')
profile_compare('recency_days')
profile_compare('video_bool')
profile_compare('recipe_bool')
print()
profile_compare('City', is_cat=True)
profile_compare('Gender', is_cat=True)
profile_compare('Device', is_cat=True)
profile_compare('Platform', is_cat=True)

print()
print('Sample sizes:')
print(f'   Champions: {len(champions)} users')
print(f'   Lost Users: {len(lost_users)} users')

In [ ]:
# Heatmap: Avg Spend by City × Age Group
pivot_spend = df.pivot_table(
    values='Total Spend (₹)',
    index='City',
    columns='age_group',
    aggfunc='mean',
    observed=True
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Cross-Segment Analysis', fontsize=15, fontweight='bold')

# Heatmap: Avg Spend by City × Age Group
sns.heatmap(pivot_spend, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, ax=axes[0])
axes[0].set_title('Avg Total Spend (₹) — City × Age Group')
axes[0].set_xlabel('Age Group')
axes[0].set_ylabel('City')

# Heatmap: Champions % by City × Engagement
pivot_champ = df.pivot_table(
    values='video_bool',
    index='City',
    columns='RFM_label',
    aggfunc='mean',
    observed=True
) * 100

pivot_champ = pivot_champ.reindex(columns=segment_order)
sns.heatmap(pivot_champ, annot=True, fmt='.0f', cmap='Blues',
            linewidths=0.5, ax=axes[1])
axes[1].set_title('Video Watch Rate (%) — City × RFM Segment')
axes[1].set_xlabel('RFM Segment')
axes[1].set_ylabel('City')

plt.tight_layout()
plt.savefig('cross_segment_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 6 - Retention & Churn Analysis

Peppo's core metric is *retention beyond week 2*. Let's use `Orders` and `recency_days` as proxies to:
1. Identify churned vs retained users
2. Understand what distinguishes retained users
3. Estimate what intervention could move the needle

In [ ]:
# Retention definition
# Retained = more than 2 orders AND active within last 30 days
# One-time = exactly 1 order
# Churned  = last active >30 days ago with ≤2 orders
# Zero-order = never ordered (acquisition failure)

def retention_label(row):
    # Users who have placed zero orders
    if row['Orders'] == 0:
        return 'Zero Order'
    # Users who have placed one order and was active > 30 days
    elif row['Orders'] == 1 and row['recency_days'] > 30:
        return 'Churned (1-time)'
    # Users who have placed <= 2 orders and was active > 30 days
    elif row['Orders'] <= 2 and row['recency_days'] > 30:
        return 'Churned (2-order)'
    # Users who have placed > 2 orders and active < 30 days
    elif row['Orders'] > 2:
        return 'Retained'
    else:
        return 'Active (early stage)'

df['retention_status'] = df.apply(retention_label, axis=1)

ret_counts = df['retention_status'].value_counts()
print('=== Retention Status Breakdown ===')
for status, count in ret_counts.items():
    pct = count / len(df) * 100
    bar = '█' * int(pct / 2)
    print(f'  {status:<22} {count:>4} ({pct:>5.1f}%) {bar}')

In [ ]:
# Retention drivers analysis
print('=== Feature Engagement Rate by Retention Status ===')
ret_eng = df.groupby('retention_status')[['video_bool', 'recipe_bool']].mean() * 100
ret_eng.columns = ['Video Watch %', 'Recipe Save %']
print(ret_eng.round(1).to_string())

print()
print('=== Avg Orders & Spend by Retention Status ===')
print(df.groupby('retention_status')[['Orders','Total Spend (₹)']].mean().round(2).to_string())

In [ ]:
# Retention funnel visualisation
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Retention Analysis', fontsize=15, fontweight='bold')

# Plot 1: Retention status donut chart
ax = axes[0]
colors_ret = ['#1A936F', '#C84B31', '#F4A261', '#004E89', '#264653']
wedges, texts, autotexts = ax.pie(
    ret_counts.values, labels=ret_counts.index,
    autopct='%1.1f%%', startangle=140,
    colors=colors_ret[:len(ret_counts)],
    wedgeprops=dict(width=0.6, edgecolor='white', linewidth=2)
)
for t in autotexts: t.set_fontsize(9)
ax.set_title('User Retention Status Distribution')

# Plot 2: Engagement rate by retention status
ax = axes[1]
ret_order = ['Retained', 'Active (early stage)', 'Churned (2-order)', 'Churned (1-time)', 'Zero Order']
ret_eng_sorted = ret_eng.reindex([r for r in ret_order if r in ret_eng.index])
x = np.arange(len(ret_eng_sorted))
w = 0.35
ax.bar(x - w/2, ret_eng_sorted['Video Watch %'],   width=w, label='Video Watch %',  color='#004E89')
ax.bar(x + w/2, ret_eng_sorted['Recipe Save %'], width=w, label='Recipe Save %', color='#FF6B35')
ax.set_xticks(x)
ax.set_xticklabels(ret_eng_sorted.index, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Engagement Rate (%)')
ax.set_title('Feature Engagement by Retention Status')
ax.legend()

plt.tight_layout()
plt.savefig('retention_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Orders distribution - retained vs churned
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Orders distribution
ax = axes[0]
df[df['retention_status'] == 'Retained']['Orders'].hist(
    bins=8, ax=ax, alpha=0.7, label='Retained', color='#1A936F')
df[df['retention_status'].str.startswith('Churned')]['Orders'].hist(
    bins=8, ax=ax, alpha=0.7, label='Churned', color='#C84B31')
ax.set_title('Order Count Distribution: Retained vs Churned')
ax.set_xlabel('Number of Orders')
ax.set_ylabel('Users')
ax.legend()

# Spend distribution
ax = axes[1]
df[df['retention_status'] == 'Retained']['Total Spend (₹)'].hist(
    bins=15, ax=ax, alpha=0.7, label='Retained', color='#1A936F')
df[df['retention_status'].str.startswith('Churned')]['Total Spend (₹)'].hist(
    bins=15, ax=ax, alpha=0.7, label='Churned', color='#C84B31')
ax.set_title('Spend Distribution: Retained vs Churned')
ax.set_xlabel('Total Spend (₹)')
ax.set_ylabel('Users')
ax.legend()

plt.tight_layout()
plt.savefig('orders_spend_dist.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 7 - LTV Quadrant Analysis

The classic 2×2 quadrant - plotting Engagement (feature usage) vs. Monetization (total spend) — helps us **prioritize which user groups to invest in** and which to deprioritize.

In [ ]:
# Engagement score (0-2 based on video + recipe)
df['engagement_score'] = df['video_bool'] + df['recipe_bool']

# Median cutoffs for quadrant
med_spend = df['Total Spend (₹)'].median()
med_eng   = df['engagement_score'].median()

def quadrant(row):
    hi_spend = row['Total Spend (₹)'] >= med_spend
    hi_eng   = row['engagement_score'] >= med_eng
    if hi_spend and hi_eng:
        return 'Stars (High $, High Eng)'
    elif hi_spend and not hi_eng:
        return 'Whales (High $, Low Eng)'
    elif not hi_spend and hi_eng:
        return 'Enthusiasts (Low $, High Eng)'
    else:
        return 'At Risk (Low $, Low Eng)'

df['ltv_quadrant'] = df.apply(quadrant, axis=1)

quad_summary = df.groupby('ltv_quadrant').agg(
    Users      = ('User ID', 'count'),
    Avg_Orders = ('Orders', 'mean'),
    Avg_Spend  = ('Total Spend (₹)', 'mean')
).round(2)

print('=== LTV Quadrant Summary ===')
print(quad_summary.to_string())

In [ ]:
# Scatter quadrant plot
fig, ax = plt.subplots(figsize=(12, 8))

quad_colors = {
    'Stars (High $, High Eng)':     '#1A936F',
    'Whales (High $, Low Eng)':     '#004E89',
    'Enthusiasts (Low $, High Eng)': '#F4A261',
    'At Risk (Low $, Low Eng)':     '#C84B31',
}

for quad, grp in df.groupby('ltv_quadrant'):
    ax.scatter(
        grp['engagement_score'] + np.random.uniform(-0.05, 0.05, len(grp)),
        grp['Total Spend (₹)'],
        alpha=0.4, s=30,
        color=quad_colors.get(quad, 'gray'),
        label=f'{quad} (n={len(grp)})'
    )

ax.axhline(med_spend, color='gray', linestyle='--', linewidth=1.5, alpha=0.6)
ax.axvline(med_eng,   color='gray', linestyle='--', linewidth=1.5, alpha=0.6)

# Quadrant labels
offset = 50
ax.text(1.7, med_spend + offset, 'Stars',         fontsize=12, color='#1A936F', fontweight='bold')
ax.text(0.1, med_spend + offset, 'Whales',        fontsize=12, color='#004E89', fontweight='bold')
ax.text(1.7, med_spend - 150,    'Enthusiasts',   fontsize=12, color='#F4A261', fontweight='bold')
ax.text(0.1, med_spend - 150,    'At Risk',       fontsize=12, color='#C84B31', fontweight='bold')

ax.set_xlabel('Engagement Score (0 = none, 1 = video or recipe, 2 = both)', fontsize=11)
ax.set_ylabel('Total Spend (₹)', fontsize=11)
ax.set_title('LTV Quadrant: Engagement vs Monetization', fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=9, framealpha=0.8)
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['No Engagement', 'Video or Recipe', 'Both'], fontsize=10)

plt.tight_layout()
plt.savefig('ltv_quadrant.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 8 - Correlation & Feature Importance

Before making recommendations, let's understand which variables correlate most strongly with high orders and spend. This helps us distinguish *signal from noise* when deciding what to act on.

In [ ]:
# Encode categoricals for correlation
df_corr = df.copy()
df_corr['is_android']    = (df_corr['Device']   == 'Android').astype(int)
df_corr['is_app']        = (df_corr['Platform'] == 'App').astype(int)
df_corr['is_female']     = (df_corr['Gender']   == 'F').astype(int)
df_corr['is_tier1_top']  = df_corr['City'].isin(['Bangalore','Mumbai','Delhi']).astype(int)

corr_cols = [
    'Age', 'is_android', 'is_app', 'is_female', 'is_tier1_top',
    'video_bool', 'recipe_bool', 'engagement_score',
    'Orders', 'Total Spend (₹)'
]

corr_matrix = df_corr[corr_cols].corr()

# Focus on correlations with Orders and Spend
target_corr = corr_matrix[['Orders', 'Total Spend (₹)']].drop(['Orders','Total Spend (₹)'])
print('=== Correlations with Orders & Total Spend ===')
print(target_corr.round(3).sort_values('Orders', ascending=False).to_string())

In [ ]:
# Correlation heatmap
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Correlation Analysis', fontsize=15, fontweight='bold')

# Full correlation heatmap
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, linewidths=0.5,
    ax=axes[0], annot_kws={'size': 8}
)
axes[0].set_title('Feature Correlation Matrix')
axes[0].tick_params(axis='x', rotation=45)

# Bar chart of correlations with Orders
order_corr = target_corr['Orders'].sort_values()
colors_bar = ['#C84B31' if v < 0 else '#1A936F' for v in order_corr.values]
axes[1].barh(order_corr.index, order_corr.values, color=colors_bar)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Correlation with Order Count')
axes[1].set_xlabel('Pearson Correlation')
for i, (idx, val) in enumerate(order_corr.items()):
    axes[1].text(val + (0.005 if val >= 0 else -0.005), i,
                 f'{val:.3f}', va='center',
                 ha='left' if val >= 0 else 'right', fontsize=9)

plt.tight_layout()
plt.savefig('correlation_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 9 - Executive Dashboard

As a good PM communicates data visually. Let's build a single-page executive summary that Aditi could bring to the CEO.

In [ ]:
# Executive Dashboard
fig = plt.figure(figsize=(20, 14))
fig.patch.set_facecolor('#1C2333')
gs = GridSpec(3, 4, figure=fig, hspace=0.5, wspace=0.4)

DARK  = '#1C2333'
TEXT  = '#F0F4FF'
GREEN = '#1A936F'
BLUE  = '#4A90D9'
ORG   = '#FF6B35'
RED   = '#E05252'

fig.text(0.5, 0.97, 'PEPPO — User Growth & Retention Dashboard',
         ha='center', fontsize=18, fontweight='bold', color=TEXT)
fig.text(0.5, 0.945, 'Q1 2024 Cohort • 1,000 Users • Analysis by Aditi Rao (PM)',
         ha='center', fontsize=11, color='#9BAAC0')

def dark_ax(ax, title=''):
    ax.set_facecolor('#252E42')
    ax.tick_params(colors=TEXT, labelsize=9)
    ax.xaxis.label.set_color(TEXT)
    ax.yaxis.label.set_color(TEXT)
    ax.spines[:].set_visible(False)
    if title:
        ax.set_title(title, color=TEXT, fontsize=10, fontweight='bold', pad=8)

# KPI Cards (top row)
kpis = [
    ('Total Users', '1,000', '#4A90D9'),
    ('Avg Orders',  f"{df['Orders'].mean():.1f}", '#1A936F'),
    ('Avg LTV (₹)', f"₹{df['Total Spend (₹)'].mean():,.0f}", '#FF6B35'),
    ('Retention Rate', f"{(df['retention_status']=='Retained').mean()*100:.0f}%", '#E05252'),
]
for i, (label, value, col) in enumerate(kpis):
    ax = fig.add_subplot(gs[0, i])
    ax.set_facecolor(col)
    ax.set_xticks([]); ax.set_yticks([])
    ax.spines[:].set_visible(False)
    ax.text(0.5, 0.6, value, ha='center', va='center',
            fontsize=26, fontweight='bold', color='white', transform=ax.transAxes)
    ax.text(0.5, 0.2, label, ha='center', va='center',
            fontsize=11, color='white', transform=ax.transAxes, alpha=0.9)

# Row 2, Col 0-1: RFM pie
ax = fig.add_subplot(gs[1, 0:2])
dark_ax(ax, 'RFM Segment Distribution')
rfm_c = df['RFM_label'].value_counts().reindex(segment_order)
wedges, texts, autos = ax.pie(
    rfm_c.values,
    labels=rfm_c.index,
    autopct='%1.0f%%',
    colors=['#1A936F','#4A90D9','#F4A261','#E05252'],
    wedgeprops={'edgecolor': DARK, 'linewidth': 2},
    startangle=90
)
for t in texts: t.set_color(TEXT)
for a in autos: a.set_color('white'); a.set_fontsize(9)

# Row 2, Col 2-3: Engagement group bar
ax = fig.add_subplot(gs[1, 2:4])
dark_ax(ax, 'Avg Orders by Engagement Type')
eng_order = df.groupby('engagement_group')['Orders'].mean().sort_values(ascending=False)
colors_eng = [GREEN, BLUE, ORG, RED]
bars = ax.bar(eng_order.index, eng_order.values, color=colors_eng)
ax.set_xticklabels(eng_order.index, color=TEXT, rotation=10, ha='right')
for bar, val in zip(bars, eng_order.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.2f}', ha='center', color=TEXT, fontsize=9)

# Row 3, Col 0: City avg spend
ax = fig.add_subplot(gs[2, 0])
dark_ax(ax, 'Avg LTV by City')
city_s = df.groupby('City')['Total Spend (₹)'].mean().sort_values()
ax.barh(city_s.index, city_s.values, color=BLUE)
ax.set_xticklabels([])
for bar, val in zip(ax.patches, city_s.values):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f'₹{val:.0f}', va='center', color=TEXT, fontsize=8)

# Row 3, Col 1: Age group orders
ax = fig.add_subplot(gs[2, 1])
dark_ax(ax, 'Avg Orders by Age Group')
age_o = df.groupby('age_group', observed=True)['Orders'].mean()
ax.bar(age_o.index, age_o.values, color=ORG)
ax.set_xticklabels(age_o.index, color=TEXT, rotation=20, ha='right')

# Row 3, Col 2: Platform orders
ax = fig.add_subplot(gs[2, 2])
dark_ax(ax, 'Avg Orders: App vs Web')
plat_o = df.groupby('Platform')['Orders'].mean()
ax.bar(plat_o.index, plat_o.values, color=[GREEN, RED])
ax.set_xticklabels(plat_o.index, color=TEXT)
for bar, val in zip(ax.patches, plat_o.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.2f}', ha='center', color=TEXT, fontsize=10)

# Row 3, Col 3: Orders histogram
ax = fig.add_subplot(gs[2, 3])
dark_ax(ax, 'Order Count Distribution')
ax.hist(df['Orders'], bins=8, color=BLUE, edgecolor=DARK)

plt.savefig('peppo_executive_dashboard.png', dpi=150, bbox_inches='tight',
            facecolor=DARK)
plt.show()

---
## Section 10 - My Recommendations

### Priority Segment 1: "Video-First Millennial" (Age 25–34, Android App, Engaged)

**Who they are:** Users aged 25–34 on the Android app who watched at least one video. They represent the core of Peppo's Champions segment.

**Data support:**
- Video watchers order significantly more than non-watchers
- Both-engaged users (video + recipe save) have the highest LTV
- Android App is the dominant platform (80% of users)

**Recommended actions:**
1. **Activation experiment:** In-app nudge to watch the first video within 24 hours of signup, and test "Watch & Cook" as onboarding step 2
2. **Recipe Save prompt:** After video completion, prompt to save the recipe, this deepens engagement loop
3. **Expand recipe catalog:** 62% of survey respondents cited "not enough variety", so, this is the #1 friction point for this segment

---

### Priority Segment 2: "Whales" (High Spend, Low Engagement)

**Who they are:** Users with above-median spend but low feature engagement. They buy but don't deeply use the product, so they are at risk of churning to competitors.

**Data support:**
- High spend despite 0 engagement suggests price-insensitive users ordering for convenience
- No video/recipe usage = no stickiness from Peppo's core differentiator

**Recommended actions:**
1. **Personalized nudge:** Email/push campaign showcasing recipes matching their order history
2. **Loyalty tier upgrade path:** Create a path from "Whales" into a higher loyalty tier with exclusive content
3. **Interview 10 Whales:** Qualitative research to understand why they buy without engaging

---

### What to Deprioritize

| Decision | Rationale |
|---|---|
| Don't expand to Tier-2 cities yet | Tier-1 retention is unsolved; Tier-2 would face same drop-off at higher CAC |
| Don't remove single-serve kits immediately | Need data on whether singles correlate with champions before cutting |
| Discount campaigns (Growth Lead's suggestion) | Discounts attract low-intent users and worsen LTV, so, not the fix |

---

### Experiments to Run

| Experiment | Hypothesis | Metric |
|---|---|---|
| Onboarding video gate | First-order users who watch video retain at higher rates | 30-day retention rate |
| Recipe save prompt post-video | Prompting save increases reorder probability | Recipe save rate + 2nd order % |
| Variety expansion | More recipes reduce "not enough variety" churn | Survey score + monthly active users |
| Price test on family kits | Higher-priced family kits have better margins | Avg order value + margin |

---

In [ ]:
# Final segment summary table
print('═' * 65)
print('   PEPPO STRATEGIC SEGMENT SUMMARY')
print('═' * 65)

summary = df.groupby('RFM_label').agg(
    Users          = ('User ID', 'count'),
    Avg_Orders     = ('Orders', 'mean'),
    Avg_Spend_INR  = ('Total Spend (₹)', 'mean'),
    Video_Rate_pct = ('video_bool', lambda x: x.mean() * 100),
    Recipe_Rate_pct= ('recipe_bool', lambda x: x.mean() * 100),
).round(1).reindex(segment_order)

summary.columns = ['Users', 'Avg Orders', 'Avg LTV (₹)', 'Video %', 'Recipe Save %']
print(summary.to_string())

print()
print('Key Takeaway for CEO Presentation:')
print('   Champions (top segment) watch videos at higher rates and save more recipes.')
print('   Getting more users to the "first video" is Peppo\'s activation lever.')
print('   Prioritize: Video onboarding → Recipe catalog expansion → Loyalty deepening.')
print()
print('Analysis complete. See saved PNG files for presentation assets.')

*Notebook built as part of NextLeap PM Fellowship*

*Happy Learning!*